# 01 — Cohort and municipality-level indicators

Builds the analysis base (Methods §*Data sources*, §*Study population and outcome measures*):

1. Load the neonatal cohort (SIM); exclude deaths with unknown municipality of residence.
2. Group the underlying cause of death into four **action groups** (where intervention would act) plus
   an ill-defined residual.
3. Aggregate to the municipality of residence, join live births (SINASC), compute the neonatal
   mortality rate (NMR).

Output: `data/processed/muni_base_2014_2024.csv`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
pd.set_option('display.width', 160)
RAW='../data/raw'; PROC='../data/processed'


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "~/Library/Python/3.9/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "~/Library

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "~/Library/Python/3.9/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "~/Library

AttributeError: _ARRAY_API not found

## 1. Load and verify the neonatal cohort (SIM)

Codes ending in `0000` are state-level "unknown municipality" placeholders (unknown residence), not
real municipalities, and are excluded from this municipality-level study.

In [2]:
sim = pd.read_csv(f'{RAW}/coorte_neonatal_2014_2024.csv', sep=';', dtype=str,
                  usecols=['CODMUNRES','CAUSABAS','TIPOBITO','DTOBITO','DTNASC','IDADE_DIAS'])
sim['CODMUNRES'] = sim['CODMUNRES'].str[:6]
n_raw = len(sim)
unknown = sim['CODMUNRES'].str.endswith('0000')
print(f'raw records                    : {n_raw:,}')
print(f'unknown-residence (…0000) drop : {unknown.sum():,}')
sim = sim[~unknown].copy()

sim['year']     = sim['DTOBITO'].str[-4:].astype(int)
sim['age_days'] = pd.to_numeric(sim['IDADE_DIAS'], errors='coerce')
print(f'analysed deaths                : {len(sim):,}')
print(f'years                          : {sim.year.min()}-{sim.year.max()}')
print(f'TIPOBITO (2 = liveborn)        : {sorted(sim.TIPOBITO.unique())}')
print(f'age_days range                 : {int(sim.age_days.min())}-{int(sim.age_days.max())} | >27d: {(sim.age_days>27).sum()}')
print(f'municipalities of residence    : {sim.CODMUNRES.nunique():,}')

raw records                    : 260,219
unknown-residence (…0000) drop : 196
analysed deaths                : 260,023
years                          : 2014-2024
TIPOBITO (2 = liveborn)        : ['2']
age_days range                 : 0-27 | >27d: 0
municipalities of residence    : 5,481


## 2. Group the underlying cause into four action groups

- **prenatal** — reducible by care during pregnancy: maternal factors/pregnancy (P00–P02, P04),
  prematurity/growth (P05, P07, P08), haemolytic disease (P55), congenital syphilis (A50).
- **delivery** — reducible by care during labour/birth: labour–delivery complications (P03),
  birth trauma (P10–P15), intrapartum hypoxia/asphyxia (P20–P21).
- **newborn** — reducible by newborn care: respiratory (P22–P28), infections (P35–P39) and other
  newborn conditions (P29, P50–P54, P56–P91, …).
- **malformation** — congenital malformations (Q00–Q99), not reducible by SUS interventions.
- **illdef** — ill-defined / not attributable (P96, ICD-10 R codes, external causes): a small residual,
  excluded from the action-group composition.

In [3]:
GROUPS = ['prenatal','delivery','newborn','malformation','illdef']
ACTION = ['prenatal','delivery','newborn','malformation']   # composition (illdef excluded)

def cause_group(code):
    if pd.isna(code) or len(code) < 3: return 'illdef'
    if code[:3].upper() == 'A50': return 'prenatal'          # congenital syphilis
    L = code[0].upper()
    try: n = int(code[1:3])
    except: return 'illdef'
    if L == 'Q': return 'malformation'
    if L == 'P':
        if n in (0,1,2,4,5,7,8,55): return 'prenatal'
        if n == 3 or 10 <= n <= 15 or n in (20,21): return 'delivery'
        if n == 96: return 'illdef'
        return 'newborn'
    return 'illdef'                                          # non-P/Q (R, external, …)

sim['cause_group'] = sim['CAUSABAS'].map(cause_group)
dist = sim['cause_group'].value_counts().reindex(GROUPS)
print(dist.to_frame('deaths').assign(pct=(dist/len(sim)*100).round(1)).to_string())

              deaths   pct
cause_group               
prenatal       73598  28.3
delivery       19971   7.7
newborn        97607  37.5
malformation   54620  21.0
illdef         14227   5.5


## 3. Municipality-level aggregation, live births and NMR

In [4]:
deaths = (sim.pivot_table(index='CODMUNRES', columns='cause_group',
                          values='CAUSABAS', aggfunc='size', fill_value=0)
             .reindex(columns=GROUPS, fill_value=0))
deaths['deaths_total'] = deaths[GROUPS].sum(axis=1)
deaths = deaths.reset_index()

births = pd.read_csv(f'{PROC}/nascidos_muni_ano.csv', dtype={'CODMUNRES': str})
births['CODMUNRES'] = births['CODMUNRES'].str[:6]
births = (births.groupby('CODMUNRES', as_index=False)['nascimentos'].sum()
                .rename(columns={'nascimentos': 'live_births'}))

base = deaths.merge(births, on='CODMUNRES', how='left')
base['live_births'] = base['live_births'].fillna(0).astype(int)
base['nmr'] = np.where(base['live_births'] > 0,
                       base['deaths_total'] / base['live_births'] * 1000, np.nan)

national_nmr = len(sim) / births['live_births'].sum() * 1000
print(f'municipalities with >=1 death : {len(base):,}')
print(f'  with valid NMR (births>0)   : {(base.live_births>0).sum():,}')
print(f'national NMR                  : {national_nmr:.2f} per 1,000 live births')

municipalities with >=1 death : 5,481
  with valid NMR (births>0)   : 5,481
national NMR                  : 8.54 per 1,000 live births


## 4. Save analysis base

In [5]:
cols = ['CODMUNRES'] + GROUPS + ['deaths_total', 'live_births', 'nmr']
base[cols].to_csv(f'{PROC}/muni_base_2014_2024.csv', index=False)
print('saved -> data/processed/muni_base_2014_2024.csv', base[cols].shape)
base[cols].head()

saved -> data/processed/muni_base_2014_2024.csv (5481, 9)


,CODMUNRES,prenatal,delivery,newborn,malformation,illdef,deaths_total,live_births,nmr
0,110001,4,2,10,7,3,26,3903,6.661542
1,110002,31,11,76,32,7,157,17663,8.888637
2,110003,1,1,3,5,1,11,801,13.732834
3,110004,31,6,47,26,6,116,14922,7.773757
4,110005,6,2,9,3,1,21,2876,7.301808
